In [1]:
!pip install -q google-genai pandas

In [2]:
from google.colab import userdata
from google import genai
import pandas as pd

MODEL = "gemini-3.1-flash-lite"

client = genai.Client(
    api_key=userdata.get("GEMINI_API_KEY"),
    http_options={"timeout": 30000}
)

print("Gemini API connected successfully!")

Gemini API connected successfully!


In [3]:
SYSTEM = "You are a course assistant for Generative AI. Answer briefly and cite the course handout."

CONTEXT = "Module III covers embeddings, chunking, MongoDB, Chroma and RAG evaluation with Hit Rate@k and MRR. " * 8

questions = [
    "What is RAG?",
    "Explain MRR with an example.",
    "రాగ్ అంటే ఏమిటి?",
    "आरएजी क्या है?"
]

rows = []

for q in questions:
    r = client.models.generate_content(
        model=MODEL,
        contents=SYSTEM + "\n" + CONTEXT + "\nQuestion: " + q
    )

    rows.append({
        "question": q,
        "input_tokens": r.usage_metadata.prompt_token_count,
        "output_tokens": r.usage_metadata.candidates_token_count
    })

sample = pd.DataFrame(rows)

print(sample.to_string(index=False))

                    question  input_tokens  output_tokens
                What is RAG?           221             67
Explain MRR with an example.           223            140
            రాగ్ అంటే ఏమిటి?           222             88
              आरएजी क्या है?           222             60


In [4]:
USERS_PER_DAY = 1000
QUESTIONS_PER_USER = 5

PEAK_HOUR_SHARE = 0.20

PRICE_IN = 0.10
PRICE_OUT = 0.40

RPM_LIMIT = 15
RPD_LIMIT = 1000

CONTEXT_LIMIT = 1000000

print("Budget assumptions configured successfully!")

Budget assumptions configured successfully!


In [5]:
avg_in = sample.input_tokens.mean()
avg_out = sample.output_tokens.mean()

req_day = USERS_PER_DAY * QUESTIONS_PER_USER

cost_req = (avg_in * PRICE_IN + avg_out * PRICE_OUT) / 1e6

peak_rpm = req_day * PEAK_HOUR_SHARE / 60

budget = {
    "avg input tokens / request": round(avg_in),
    "avg output tokens / request": round(avg_out),
    "requests / day": req_day,
    "tokens / day": round(req_day * (avg_in + avg_out)),
    "cost / request (USD)": round(cost_req, 6),
    "cost / day (USD)": round(cost_req * req_day, 2),
    "cost / month (USD)": round(cost_req * req_day * 30, 2),
    "cost / month (INR @ 84)": round(cost_req * req_day * 30 * 84),
    "peak requests / minute": round(peak_rpm, 1),
    "fits free-tier RPM?": peak_rpm <= RPM_LIMIT,
    "fits free-tier RPD?": req_day <= RPD_LIMIT,
    "context used per request": f"{avg_in / CONTEXT_LIMIT:.4%}"
}

print("COST BUDGET REPORT")

for k, v in budget.items():
    print(f"{k:28s}: {v}")

COST BUDGET REPORT
avg input tokens / request  : 222
avg output tokens / request : 89
requests / day              : 5000
tokens / day                : 1553750
cost / request (USD)        : 5.8e-05
cost / day (USD)            : 0.29
cost / month (USD)          : 8.65
cost / month (INR @ 84)     : 727
peak requests / minute      : 16.7
fits free-tier RPM?         : False
fits free-tier RPD?         : False
context used per request    : 0.0222%


In [6]:
if peak_rpm <= RPM_LIMIT and req_day <= RPD_LIMIT:
    print("Decision: The system fits within the configured free-tier limits.")
else:
    print("Decision: Free-tier limits are insufficient.")
    print("Recommended solutions:")
    print("1. Add caching")
    print("2. Use a paid tier")
    print("3. Implement a request queue")

Decision: Free-tier limits are insufficient.
Recommended solutions:
1. Add caching
2. Use a paid tier
3. Implement a request queue
